# TOPIC 10 – ĐO LƯỜNG RỦI RO DANH MỤC VN30 BẰNG DCC-GARCH
## Thành viên 1 – Trang: Data pipeline & EDA (Stylized Facts)
Chạy **đầu tiên**. Chạy bằng **Restart Kernel → Run All**. Thứ tự chung của cả nhóm: TV1 → TV2 → TV3 → TV4 → TV5.

| | |
|---|---|
| Đọc từ | 5 file Excel giá trong `input/` (FPT, HPG, MWG, VCB, VHM) |
| Ghi ra | `TV1_Data_EDA/` (`log_returns.csv`, `panel_AdjClose.csv`, hình, `TV1_statistical_tests.xlsx`) |
| Cấu hình | ô 0.2 (đường dẫn, hằng số) – **giống hệt ở cả 5 notebook**, chỉ cần sửa `PROJECT_PATH` / `OUTPUT_PATH` cho đúng máy |

Thư viện: numpy, pandas, scipy, matplotlib, openpyxl (statsmodels tuỳ chọn). Xem `README.md`.

In [ ]:
# ===================== 0.1 THƯ VIỆN =====================
import os                     # đọc biến môi trường
import re                     # biểu thức chính quy: tách dòng cổ tức / chia tách trong file Yahoo
import warnings               # tắt cảnh báo không cần thiết
import datetime as dt         # kiểu ngày tháng
from itertools import combinations   # tạo các cặp cổ phiếu
from pathlib import Path      # thao tác đường dẫn an toàn trên Windows

import numpy as np            # tính toán ma trận
import pandas as pd           # bảng dữ liệu
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from scipy import stats                     # phân phối Normal / t / chi-bình phương, JB
from scipy.optimize import minimize         # tối đa hoá hàm hợp lý (MLE)
from scipy.signal import lfilter            # lọc tuyến tính nhanh cho phương trình GARCH
from scipy.special import gammaln           # log Gamma cho mật độ Student-t

try:                                        # statsmodels: chỉ dùng cho ADF / KPSS (nếu có)
    from statsmodels.tsa.stattools import adfuller, kpss
    from statsmodels.tools.sm_exceptions import InterpolationWarning
    HAS_SM = True
except ImportError:
    HAS_SM = False

try:                                        # display() có sẵn trong Jupyter; chạy file .py thì thay bằng print
    display
except NameError:
    display = print

warnings.filterwarnings("ignore", category=RuntimeWarning)
pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 30)
plt.rcParams["figure.dpi"] = 100
print("statsmodels có sẵn:", HAS_SM)

In [ ]:
# ===================== 0.2 CẤU HÌNH DÙNG CHUNG CHO CẢ 5 THÀNH VIÊN =====================
# >>> ĐƯỜNG DẪN – sửa 2 dòng này nếu đổi máy / đổi thư mục <<<
PROJECT_PATH = r"D:\Năm 3\Ứng dụng công nghệ trong TCNH\bản demo"          # thư mục dự án (chứa input/, ouput/, code/)
OUTPUT_PATH = r"D:\Năm 3\Ứng dụng công nghệ trong TCNH\bản demo\ouput"    # nơi lưu toàn bộ kết quả


def find_base_dir() -> Path:
    """Tìm thư mục dự án: biến môi trường PROJECT_DIR -> PROJECT_PATH -> thư mục cha của code/."""
    env = os.environ.get("PROJECT_DIR")
    if env:
        return Path(env)
    if Path(PROJECT_PATH).exists():
        return Path(PROJECT_PATH)
    cwd = Path.cwd()
    return cwd.parent if cwd.name.lower() == "code" else cwd


def pick_dir(base: Path, names) -> Path:
    """Chọn thư mục con đã tồn tại (chấp nhận Input/input)."""
    return next((base / n for n in names if (base / n).exists()), base / names[0])


BASE_DIR = find_base_dir()
INPUT_DIR = pick_dir(BASE_DIR, ["input", "Input"])
if not list(INPUT_DIR.glob("*.xlsx")):                  # dự phòng: thư mục Input cũ của Project midterm
    old_in = Path(r"D:\Năm 3\Ứng dụng công nghệ trong TCNH\Project midterm\Input")
    if list(old_in.glob("*.xlsx")):
        INPUT_DIR = old_in
# Output: biến môi trường OUTPUT_DIR -> OUTPUT_PATH (nếu ổ D: tồn tại) -> BASE_DIR/ouput
OUTPUT_DIR = Path(os.environ.get("OUTPUT_DIR") or
                  (OUTPUT_PATH if Path(OUTPUT_PATH).anchor and Path(Path(OUTPUT_PATH).anchor).exists()
                   else BASE_DIR / "ouput"))
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
ROOT_OUT = OUTPUT_DIR                                    # mỗi thành viên một thư mục con trong ouput/
OUT = {k: ROOT_OUT / v for k, v in {"TV1": "TV1_Data_EDA", "TV2": "TV2_GARCH", "TV3": "TV3_DCC",
                                     "TV4": "TV4_VaR_Backtest", "TV5": "TV5_Euler_Stress"}.items()}
for d in OUT.values():
    d.mkdir(parents=True, exist_ok=True)

# ---- Hằng số dùng chung ----
TICKERS_ORDER = ["FPT", "HPG", "MWG", "VCB", "VHM"]    # thứ tự hiển thị (mã khác trong Input vẫn được đọc)
ALPHA = 0.05                 # mức ý nghĩa của mọi kiểm định
LB_LAGS = [5, 10, 20]        # số trễ Ljung-Box
TRADING_DAYS = 252           # số phiên / năm để năm hoá
SCALE = 100.0                # ước lượng GARCH trên lợi suất % (r × 100) cho ổn định số học
SELECT_BY = "BIC"            # tiêu chí chọn mô hình GARCH: "AIC" hoặc "BIC"
PAIRS = [("VCB", "VHM"), ("HPG", "FPT")]               # cặp tương quan động theo đề
VAR_LEVELS = [0.95, 0.99]    # mức tin cậy VaR
HS_WINDOW = 250              # cửa sổ Historical Simulation (≈ 1 năm)
BASEL_WINDOW = 250           # cửa sổ Basel Traffic Light (250 phiên gần nhất)
VIOL_MODEL = "Normal"        # >>> MÔ HÌNH ĐÁNH DẤU VI PHẠM TRÊN MỌI HÌNH & DIỄN GIẢI TV4: "Normal" hoặc "t" <<<
VIOL_OOS = {"Normal": "DCC-N", "t": "DCC-t"}[VIOL_MODEL]   # tên tương ứng trong phần ngoài mẫu
EULER_CL = 0.99              # mức tin cậy dùng cho phân bổ rủi ro Euler
STRESS = {                   # giai đoạn khủng hoảng theo đề (TV5)
    "Covid-19 (02–04/2020)": ("2020-02-01", "2020-04-30"),
    "Trái phiếu & lãi suất (04–11/2022)": ("2022-04-01", "2022-11-30"),
}


STRESS_COLORS = ["tab:red", "tab:orange"]   # mỗi giai đoạn một màu để phân biệt Covid-19 với Trái phiếu & lãi suất


def shade_stress(ax, alpha=0.12, label=True, xlim=None):
    """Tô màu từng giai đoạn khủng hoảng (mỗi giai đoạn một màu, chú thích theo tên giai đoạn).
    xlim=(đầu, cuối): bỏ qua giai đoạn nằm hoàn toàn ngoài khoảng này (tránh kéo giãn trục thời gian)."""
    for (name, (s, e)), c in zip(STRESS.items(), STRESS_COLORS):
        s, e = pd.Timestamp(s), pd.Timestamp(e)
        if xlim is not None and (e < xlim[0] or s > xlim[1]):
            continue
        ax.axvspan(s, e, color=c, alpha=alpha, label=(name if label else None))


def grid_axes(n, ncol=2, h=3.2, w=13, **kw):
    """Tạo lưới ô vẽ cho n mã, ẩn các ô thừa."""
    ncol = min(ncol, n)
    nrow = int(np.ceil(n / ncol))
    fig, axes = plt.subplots(nrow, ncol, figsize=(w, h * nrow), squeeze=False, **kw)
    for ax in axes.flat[n:]:
        ax.set_visible(False)
    return fig, axes.flat


print("BASE_DIR  :", BASE_DIR)
print("INPUT_DIR :", INPUT_DIR, "| tồn tại:", INPUT_DIR.exists())
print("OUTPUT_DIR:", OUTPUT_DIR)


def period_masks(index):
    """Mặt nạ True/False cho từng giai đoạn: các giai đoạn STRESS + 'Bình thường' (ngoài mọi giai đoạn stress)."""
    m = {name: (index >= s) & (index <= e) for name, (s, e) in STRESS.items()}
    m = {"Bình thường": ~np.any(np.column_stack(list(m.values())), axis=1), **m}
    return m


---
# THÀNH VIÊN 1: DATA PIPELINE & KHÁM PHÁ DỮ LIỆU (EDA & STYLIZED FACTS)
## 1.1 – 1.2 Thu thập & làm sạch dữ liệu
5 cổ phiếu VN30 đại diện 5 ngành: **VCB** (ngân hàng), **HPG** (thép), **FPT** (công nghệ), **MWG** (bán lẻ), **VHM** (bất động sản),
giai đoạn 2019–2024 (bao trùm Covid-19 và khủng hoảng trái phiếu 2022), tải từ Yahoo Finance.

**Tầng 1 – Ý nghĩa:**
- *Adjusted Close* đã điều chỉnh chia tách cổ phiếu và cổ tức ⇒ lợi suất tính từ Adj Close là lợi suất **tổng** thực nhận của nhà đầu tư,
  không bị "rơi giá" giả vào ngày giao dịch không hưởng quyền. Hệ số điều chỉnh $f_t = \text{Adj Close}_t / \text{Close}_t$.
- *Ngày nghỉ lễ / Tết*: file Yahoo vẫn có dòng cho ngày nghỉ với Volume = 0 (hoặc "-") ⇒ không phải phiên giao dịch thật, phải loại bỏ,
  nếu không sẽ sinh ra nhiều lợi suất = 0 giả làm méo phương sai và tự tương quan. Ngoài ngày lễ, Yahoo còn **thiếu dữ liệu** một số
  ngày giao dịch bình thường; notebook đối chiếu với lịch nghỉ HOSE (`VN_HOLIDAYS`) để tách hai loại này (`non_trading_days.csv`).
- *Calendar alignment*: mô hình đa biến (DCC) cần ma trận lợi suất **cùng ngày** cho cả 5 mã ⇒ dựng lịch chuẩn chung, mã nào tạm ngừng
  giao dịch thì giữ giá phiên trước (forward-fill, tối đa 10 phiên).

**Tầng 2 – Input / Output:** Input: 5 file `.xlsx` trong `Input/` (tiêu đề cột kiểu Yahoo, số có dấu phẩy nghìn, chen dòng
"Dividend" / "Stock Splits"). Output: `prices` (DataFrame Adj Close đã căn lịch, T × 5), `clean_by_ticker/*.csv`, `cleaning_report.txt`.

**Tầng 3 – Cách đọc:** nhật ký in ra số dòng sự kiện đã tách, số ngày nghỉ lễ và số ngày thiếu dữ liệu, số phiên điền giá cho từng mã,
và bảng các ngày lợi suất vượt biên độ ±7% của HOSE kèm nguyên nhân (lợi suất dồn qua kỳ nghỉ hoặc qua ngày thiếu dữ liệu).
Toàn bộ nhật ký được ghi vào `cleaning_report.txt`; bảng ngoại lai lưu ở `outliers_check.csv`.

In [ ]:
# ===================== 1.1 ĐỌC DỮ LIỆU THÔ =====================
REQUIRED_COLS = ["Open", "High", "Low", "Close", "Adj Close", "Volume"]
FFILL_LIMIT = 10          # tối đa số phiên liên tiếp được điền giá khi cổ phiếu tạm ngừng giao dịch
GAP_THRESHOLD_DAYS = 4    # 2 phiên thật cách nhau > 4 ngày lịch => ghi nhận kỳ nghỉ dài
PRICE_LIMIT = 0.07        # biên độ dao động giá ngày của HOSE (±7%)
EVENTS = {}               # sự kiện cổ tức / chia tách tách ra từ file Yahoo
REPORT = []               # nhật ký làm sạch -> cleaning_report.txt


def log(msg=""):
    print(msg)
    REPORT.append(str(msg))


def _map_col(col):
    """Chuẩn hoá tên cột: 'Adj Close     Adjusted close price...' -> 'Adj Close'."""
    c = str(col).strip().lower()
    for key, name in [("adj", "Adj Close"), ("date", "Date"), ("open", "Open"), ("high", "High"),
                      ("low", "Low"), ("close", "Close"), ("volume", "Volume")]:
        if c.startswith(key):
            return name
    return col


def _to_number(s: pd.Series) -> pd.Series:
    """Đọc số an toàn: '1,234.56', '1.234,56', '-', '#N/A' ... -> float (ô không phải số -> NaN)."""
    if pd.api.types.is_numeric_dtype(s):
        return pd.to_numeric(s, errors="coerce")

    def conv(v):
        if v is None or (isinstance(v, float) and np.isnan(v)):
            return np.nan
        if isinstance(v, (int, float, np.number)):
            return float(v)
        t = str(v).strip().replace("\u00a0", "").replace(" ", "")
        if t in ("", "-", "--", "#N/A", "N/A", "NA", "nan", "None"):
            return np.nan
        if "," in t and "." in t:                    # dấu xuất hiện sau cùng là dấu thập phân
            t = t.replace(".", "").replace(",", ".") if t.rfind(",") > t.rfind(".") else t.replace(",", "")
        elif "," in t:
            t = t.replace(",", "") if re.fullmatch(r"-?\d{1,3}(,\d{3})+", t) else t.replace(",", ".")
        elif t.count(".") > 1:
            t = t.replace(".", "")
        try:
            return float(t)
        except ValueError:
            return np.nan
    return s.map(conv).astype(float)


def _parse_dates(s: pd.Series, swap: bool) -> pd.Series:
    """Đọc cột ngày hỗn hợp (datetime thật + chuỗi dd/mm/yyyy). swap=True: đảo ngày<->tháng cho ô Excel đã tự đổi sai."""
    out = []
    for v in s:
        if pd.isna(v):
            out.append(pd.NaT)
        elif isinstance(v, (pd.Timestamp, dt.datetime, dt.date)):
            ts = pd.Timestamp(v)
            if swap and ts.day <= 12:
                ts = pd.Timestamp(year=ts.year, month=ts.day, day=ts.month)
            out.append(ts)
        else:
            txt = str(v).strip()
            iso = re.match(r"^\d{4}-\d{2}-\d{2}", txt) is not None
            out.append(pd.to_datetime(txt, dayfirst=not iso, errors="coerce"))
    return pd.Series(pd.to_datetime(out), index=s.index)


def parse_dates(s: pd.Series, name: str) -> pd.Series:
    """Thử 2 cách đọc ngày, chọn cách cho chuỗi đúng thứ tự thời gian và ít ngày trùng nhất."""
    def score(d):
        diff = d.dropna().diff().dropna()
        order = int(min((diff < pd.Timedelta(0)).sum(), (diff > pd.Timedelta(0)).sum()))
        return order + int(d.dropna().duplicated().sum())
    cands = {False: _parse_dates(s, False)}
    if any(isinstance(v, (pd.Timestamp, dt.datetime, dt.date)) for v in s):
        cands[True] = _parse_dates(s, True)
    best = min(cands, key=lambda k: score(cands[k]))
    if best:
        log(f"  - {name}: phát hiện ngày bị Excel đảo ngày/tháng -> đã sửa")
    return cands[best]


def load_stock(path: Path) -> pd.DataFrame:
    """Đọc 1 file Excel -> DataFrame index = Date, các cột số chuẩn hoá; tách dòng sự kiện cổ tức/chia tách."""
    tk = path.stem.upper()
    df = pd.read_excel(path, sheet_name=0)
    df.columns = [_map_col(c) for c in df.columns]
    missing = [c for c in ["Date"] + REQUIRED_COLS if c not in df.columns]
    if missing:
        raise ValueError(f"{path.name}: thiếu cột {missing}")
    df = df[["Date"] + REQUIRED_COLS].copy()
    raw = df[REQUIRED_COLS].map(lambda v: "" if pd.isna(v) else str(v))   # dạng chuỗi để dò dòng sự kiện
    df["Date"] = parse_dates(df["Date"], tk)

    # Dòng sự kiện: Yahoo chen "790.5 Dividend ..." hoặc "23:20 Stock Splits ..." vào bảng giá
    is_event = raw.apply(lambda c: c.str.contains(r"dividend|stock\s*split", case=False, regex=True)).any(axis=1)
    if is_event.any():
        txt = raw.loc[is_event].apply(lambda r: next((v for v in r if re.search("ividend|plit", str(v))), ""), axis=1)
        m = txt.str.extract(r"^\s*([\d.,:]+)\s*(Dividend|Stock\s*Splits?)", flags=re.I)
        EVENTS[tk] = pd.DataFrame({"Date": df.loc[is_event, "Date"].values, "Ticker": tk,
                                   "Type": m[1].str.title().values, "Value": m[0].values}).drop_duplicates()
        n_div = int((EVENTS[tk]["Type"] == "Dividend").sum())
        log(f"      {tk}: tách {int(is_event.sum())} dòng sự kiện ({n_div} cổ tức, "
            f"{len(EVENTS[tk]) - n_div} chia tách/cổ phiếu thưởng)")
        df = df.loc[~is_event].copy()

    for c in REQUIRED_COLS:
        df[c] = _to_number(df[c])
    n_bad = int(df["Date"].isna().sum())
    df = df.dropna(subset=["Date"])
    df["Date"] = df["Date"].dt.normalize()
    n_dup = int(df["Date"].duplicated().sum())
    df = df.drop_duplicates(subset="Date", keep="last").sort_values("Date").set_index("Date")
    log(f"  - Đọc {path.name}: {len(df)} dòng | {df.index.min().date()} -> {df.index.max().date()} "
        f"| ngày lỗi bỏ: {n_bad} | ngày trùng bỏ: {n_dup}")
    return df


REPORT.clear()
files = sorted(p for p in INPUT_DIR.glob("*.xlsx") if not p.name.startswith("~$"))
if not files:
    raise FileNotFoundError(f"Không tìm thấy file .xlsx trong {INPUT_DIR}")
log("=" * 78); log("ĐỌC DỮ LIỆU"); log("=" * 78)
raw = {p.stem.upper(): load_stock(p) for p in files}
raw = {tk: raw[tk] for tk in [t for t in TICKERS_ORDER if t in raw] + [t for t in raw if t not in TICKERS_ORDER]}

In [ ]:
# ===================== 1.2a BƯỚC 1 – GIÁ ĐÓNG CỬA ĐIỀU CHỈNH =====================
log(""); log("=" * 78); log("BƯỚC 1: GIÁ ĐÓNG CỬA ĐIỀU CHỈNH (ADJUSTED CLOSE)"); log("=" * 78)
adj = {}
for tk, df in raw.items():
    df = df.copy()
    for c in ["Open", "High", "Low", "Close", "Adj Close"]:
        df.loc[df[c] <= 0, c] = np.nan                     # giá <= 0 không hợp lệ
    df["Adj_Factor"] = df["Adj Close"] / df["Close"]        # hệ số điều chỉnh f_t
    for c in ["Open", "High", "Low"]:
        df[f"Adj_{c}"] = df[c] * df["Adj_Factor"]           # điều chỉnh OHLC theo cùng hệ số
    n_chg = int((df["Adj_Factor"].pct_change().abs() > 1e-4).sum())
    bad_hl = int((df["High"] < df["Low"]).sum())
    log(f"  [{tk}] Adj_Factor: {df['Adj_Factor'].min():.4f} – {df['Adj_Factor'].max():.4f} "
        f"| số ngày hệ số thay đổi (cổ tức/chia tách): {n_chg} | High<Low: {bad_hl}")
    adj[tk] = df

In [ ]:
# ===================== 1.2b BƯỚC 2 – NGÀY NGHỈ LỄ, TẾT, KHÔNG GIAO DỊCH =====================
log(""); log("=" * 78); log("BƯỚC 2: NGÀY NGHỈ LỄ, TẾT & NGÀY KHÔNG GIAO DỊCH"); log("=" * 78)
trading = {}
for tk, df in adj.items():
    df = df[df.index.dayofweek < 5].dropna(subset=["Adj Close"]).copy()   # bỏ T7/CN và dòng thiếu giá
    df["Volume"] = df["Volume"].fillna(0)
    df["Is_Traded"] = df["Volume"] > 0                                     # Volume = 0 / "-" => ngày nghỉ
    real = df.index[df["Is_Traded"]].to_series()
    gaps = real.diff().dt.days
    long_gaps = gaps[gaps > GAP_THRESHOLD_DAYS]
    log(f"  [{tk}] dòng Volume=0 (không GD): {int((~df['Is_Traded']).sum())} "
        f"| số kỳ nghỉ dài > {GAP_THRESHOLD_DAYS} ngày: {len(long_gaps)} "
        f"(vd: " + ", ".join(f"{(d - pd.Timedelta(days=int(g))).date()}→{d.date()}"
                             for d, g in long_gaps.head(3).items()) + ")")
    trading[tk] = df

In [ ]:
# ===================== 1.2c BƯỚC 3 – CĂN CHỈNH LỊCH GIAO DỊCH =====================
log(""); log("=" * 78); log("BƯỚC 3: CĂN CHỈNH LỊCH GIAO DỊCH (CALENDAR ALIGNMENT)"); log("=" * 78)
start = max(df.index.min() for df in trading.values())       # giai đoạn chung của cả 5 mã
end = min(df.index.max() for df in trading.values())
traded_days = sorted(set().union(*[set(df.index[df["Is_Traded"]]) for df in trading.values()]))
master = pd.DatetimeIndex(traded_days)
master = master[(master >= start) & (master <= end)]          # lịch chuẩn = ngày có ít nhất 1 mã giao dịch
holidays = pd.bdate_range(master.min(), master.max()).difference(master)   # ngày T2–T6 không mã nào có giao dịch

# Lịch nghỉ lễ chính thức của HOSE 2019–2024 (Tết Dương lịch, Tết Nguyên đán, Giỗ Tổ, 30/4–1/5, Quốc khánh, kể cả ngày nghỉ bù).
# Ngày không giao dịch nằm NGOÀI danh sách này = ngày file Yahoo thiếu dữ liệu, không phải ngày nghỉ của thị trường.
VN_HOLIDAYS = pd.to_datetime([
    "2019-01-01", "2019-02-04", "2019-02-05", "2019-02-06", "2019-02-07", "2019-02-08", "2019-04-15",
    "2019-04-29", "2019-04-30", "2019-05-01", "2019-09-02",
    "2020-01-01", "2020-01-23", "2020-01-24", "2020-01-27", "2020-01-28", "2020-01-29", "2020-04-02",
    "2020-04-30", "2020-05-01", "2020-09-02",
    "2021-01-01", "2021-02-10", "2021-02-11", "2021-02-12", "2021-02-15", "2021-02-16", "2021-04-21",
    "2021-04-30", "2021-05-03", "2021-09-02", "2021-09-03",
    "2022-01-03", "2022-01-31", "2022-02-01", "2022-02-02", "2022-02-03", "2022-02-04", "2022-04-11",
    "2022-05-02", "2022-05-03", "2022-09-01", "2022-09-02",
    "2023-01-02", "2023-01-20", "2023-01-23", "2023-01-24", "2023-01-25", "2023-01-26",
    "2023-05-01", "2023-05-02", "2023-05-03", "2023-09-01", "2023-09-04",
    "2024-01-01", "2024-02-08", "2024-02-09", "2024-02-12", "2024-02-13", "2024-02-14", "2024-04-18",
    "2024-04-29", "2024-04-30", "2024-05-01", "2024-09-02", "2024-09-03",
])
is_holiday = holidays.isin(VN_HOLIDAYS)
non_trading = pd.DataFrame({"Date": holidays, "Thứ": holidays.day_name(),
                            "Phân loại": np.where(is_holiday, "Nghỉ lễ/Tết (lịch HOSE)",
                                                  "Không có dữ liệu (file Yahoo thiếu dòng/Volume=0)")})
missing_days = holidays[~is_holiday]
log(f"  Giai đoạn chung: {start.date()} -> {end.date()} | số phiên lịch chuẩn: {len(master)}")
log(f"  Ngày T2–T6 không có dữ liệu giao dịch ở cả 5 mã: {len(holidays)} = "
    f"{int(is_holiday.sum())} ngày nghỉ lễ/Tết + {len(missing_days)} ngày Yahoo thiếu dữ liệu")
if len(missing_days):
    log("    Ví dụ ngày thiếu dữ liệu: " + ", ".join(str(d.date()) for d in missing_days[:8]) + " ...")

price_cols = ["Open", "High", "Low", "Close", "Adj Close", "Adj_Open", "Adj_High", "Adj_Low", "Adj_Factor"]
aligned = {}
for tk, df in trading.items():
    a = df.reindex(master)
    a.index.name = "Date"
    a["Is_Traded"] = a["Is_Traded"].fillna(False).astype(bool)
    a["Volume"] = a["Volume"].fillna(0)
    a[price_cols] = a[price_cols].ffill(limit=FFILL_LIMIT)    # mã tạm ngừng GD: giữ giá phiên trước
    log(f"  [{tk}] phiên GD thật: {int(a['Is_Traded'].sum())}/{len(master)} "
        f"| phiên điền giá: {int((~a['Is_Traded']).sum())} | NaN còn lại: {int(a['Adj Close'].isna().sum())}")
    aligned[tk] = a

prices = pd.DataFrame({tk: a["Adj Close"] for tk, a in aligned.items()}).dropna()   # panel cân bằng
for tk in aligned:
    aligned[tk] = aligned[tk].loc[prices.index]
tickers = list(prices.columns)
N = len(tickers)
log(f"  Panel giá cuối cùng: {prices.shape[0]} phiên × {N} mã ({', '.join(tickers)})")

# ---- Chẩn đoán ngày lợi suất vượt biên độ ±7% ----
sr = prices.pct_change().iloc[1:]
rows = []
for tk in tickers:
    traded_idx = aligned[tk].index[aligned[tk]["Is_Traded"]]
    for d in sr.index[sr[tk].abs() > PRICE_LIMIT + 0.005]:
        prev = traded_idx[traded_idx < d]
        gap = (d - prev[-1]).days if len(prev) else np.nan
        if len(prev):
            n_miss = int(((missing_days > prev[-1]) & (missing_days < d)).sum())     # ngày cả thị trường thiếu dữ liệu
            n_hol = int(((VN_HOLIDAYS > prev[-1]) & (VN_HOLIDAYS < d)).sum())         # ngày nghỉ lễ
            n_own = int(((master > prev[-1]) & (master < d)).sum())                   # phiên thị trường có GD nhưng riêng mã này không
        else:
            n_miss = n_hol = n_own = 0
        parts = []                                                   # một ngày có thể có nhiều nguyên nhân cùng lúc
        if n_hol:
            parts.append(f"sau {n_hol} ngày nghỉ lễ")
        if n_miss:
            parts.append(f"qua {n_miss} ngày Yahoo thiếu dữ liệu")
        if n_own:
            parts.append(f"mã thiếu dữ liệu riêng {n_own} phiên")
        why = ("Dồn lợi suất nhiều ngày: " + "; ".join(parts)) if parts \
            else "Biến động thật / ngoại lệ biên độ (vd. ngày hưởng quyền)"
        rows.append({"Date": d.date(), "Mã": tk, "Lợi suất": round(sr.loc[d, tk], 4),
                     "Phiên GD liền trước": prev[-1].date() if len(prev) else None, "Cách phiên trước (ngày)": gap,
                     "Ngày lễ ở giữa": n_hol, "Ngày thiếu dữ liệu ở giữa": n_miss, "Phiên mã này không GD": n_own,
                     "Nguyên nhân": why})
outliers = pd.DataFrame(rows)
log(f"\n  Số ngày |lợi suất| > ±{PRICE_LIMIT:.0%} (+0,5% dung sai): {len(outliers)}")
if len(outliers):
    log(outliers.to_string(index=False))                    # ghi cả bảng vào cleaning_report.txt
    log("\n  Tổng hợp nguyên nhân:\n" + outliers["Nguyên nhân"].value_counts().to_string())

# ---- Lưu kết quả làm sạch ----
(OUT["TV1"] / "clean_by_ticker").mkdir(exist_ok=True)
for tk, a in aligned.items():
    a.to_csv(OUT["TV1"] / "clean_by_ticker" / f"{tk}_clean.csv", encoding="utf-8-sig")
prices.to_csv(OUT["TV1"] / "panel_AdjClose.csv", encoding="utf-8-sig")
if EVENTS:
    pd.concat(EVENTS.values()).sort_values(["Ticker", "Date"]).to_csv(
        OUT["TV1"] / "corporate_events.csv", index=False, encoding="utf-8-sig")
non_trading.to_csv(OUT["TV1"] / "non_trading_days.csv", index=False, encoding="utf-8-sig")   # thay cho market_holidays_detected.csv
outliers.to_csv(OUT["TV1"] / "outliers_check.csv", index=False, encoding="utf-8-sig")
(OUT["TV1"] / "cleaning_report.txt").write_text("\n".join(REPORT), encoding="utf-8")
print("✔ Đã lưu dữ liệu sạch vào", OUT["TV1"])

## 1.3 Tính Log Returns & trực quan hoá
**Tầng 1:** $r_t = \ln\left(\dfrac{P_t}{P_{t-1}}\right)$ với $P_t$ = Adj Close. Log return cộng được theo thời gian
($\sum r_t = \ln P_T/P_0$) và xấp xỉ lợi suất đơn khi biến động nhỏ ⇒ là đầu vào chuẩn cho GARCH.

**Tầng 2:** Input `prices` (T × N). Output `log_ret` ((T−1) × N, đơn vị thập phân) – **biến dùng chung cho TV2 → TV5**.

**Tầng 3:** Biểu đồ giá có xu hướng (không dừng); biểu đồ lợi suất dao động quanh 0 nhưng có những **cụm** biến động lớn
(2020, 2022) xen kẽ cụm yên tĩnh – dấu hiệu đầu tiên của volatility clustering.

In [ ]:
# ===================== 1.3 LOG RETURNS + BIỂU ĐỒ =====================
log_ret = np.log(prices / prices.shift(1)).iloc[1:]          # r_t = ln(P_t / P_{t-1})
log_ret.index.name = "Date"
T = len(log_ret)
print(f"Log return: {T} phiên × {N} mã | {log_ret.index.min().date()} -> {log_ret.index.max().date()} "
      f"| NaN: {int(log_ret.isna().sum().sum())}")
display(log_ret.head())
log_ret.to_csv(OUT["TV1"] / "log_returns.csv", encoding="utf-8-sig")

# (a) Chuỗi giá – mỗi mã một ô vì thang giá khác nhau
fig, axes = grid_axes(N, sharex=True)
for ax, tk in zip(axes, tickers):
    ax.plot(prices.index, prices[tk], lw=1, color="tab:blue")
    shade_stress(ax, label=False)
    ax.set_title(f"{tk} – Adjusted Close"); ax.set_ylabel("VND"); ax.grid(alpha=0.3)
fig.suptitle("Chuỗi giá đóng cửa điều chỉnh (Price Series)", fontsize=13, y=1.0)
plt.tight_layout(); plt.savefig(OUT["TV1"] / "1_price_series.png", dpi=150, bbox_inches="tight"); plt.show()

# (b) Giá chuẩn hoá = 100 tại ngày đầu để so sánh trên cùng trục
ax = (prices / prices.iloc[0] * 100).plot(figsize=(13, 4.5), lw=1.1)
shade_stress(ax); ax.axhline(100, color="grey", ls="--", lw=0.8)
ax.set_title("Giá chuẩn hoá (ngày đầu = 100)"); ax.grid(alpha=0.3); ax.legend(ncol=6)
plt.tight_layout(); plt.savefig(OUT["TV1"] / "1_price_normalized.png", dpi=150); plt.show()

# (c) Chuỗi lợi suất – cùng trục tung để so sánh mức biến động
ylim = log_ret.abs().max().max() * 1.05
fig, axes = grid_axes(N, sharex=True, sharey=True)
for ax, tk in zip(axes, tickers):
    ax.plot(log_ret.index, log_ret[tk], lw=0.6, color="tab:red")
    shade_stress(ax, alpha=0.08, label=False)
    ax.axhline(0, color="black", lw=0.5); ax.set_ylim(-ylim, ylim)
    ax.set_title(f"{tk} – Log Return (σ = {log_ret[tk].std():.2%}/ngày)"); ax.grid(alpha=0.3)
fig.suptitle("Chuỗi lợi suất logarit (Return Series)", fontsize=13, y=1.0)
plt.tight_layout(); plt.savefig(OUT["TV1"] / "1_return_series.png", dpi=150, bbox_inches="tight"); plt.show()

## 1.4a Thống kê mô tả
**Tầng 1:** Skewness $S = \frac{1}{n}\sum\left(\frac{r_t-\bar r}{\hat\sigma}\right)^3$ (S < 0: đuôi trái dài – lỗ lớn hay xảy ra hơn lãi lớn);
Kurtosis $K = \frac{1}{n}\sum\left(\frac{r_t-\bar r}{\hat\sigma}\right)^4$ (phân phối chuẩn K = 3; K > 3: **đuôi dày**, nhọn hơn chuẩn).

**Tầng 2:** `describe_returns(log_ret)` dùng `scipy.stats.skew / kurtosis` (bias=True – đúng công thức JB). Output bảng `desc`.

**Tầng 3:** So sánh Std năm hoá giữa các mã (mã nào rủi ro nhất); K > 3 ở mọi mã là bằng chứng ban đầu của fat tails.

In [ ]:
# ===================== 1.4a THỐNG KÊ MÔ TẢ =====================
def describe_returns(r: pd.DataFrame) -> pd.DataFrame:
    rows = {}
    for tk in r.columns:
        x = r[tk].dropna().values
        S = stats.skew(x, bias=True)
        K = stats.kurtosis(x, fisher=False, bias=True)        # kurtosis thường (chuẩn = 3)
        rows[tk] = {"N": len(x), "Mean": x.mean(), "Std": x.std(ddof=1), "Skewness": S, "Kurtosis": K,
                    "Excess Kurtosis": K - 3, "Min": x.min(), "Median": np.median(x), "Max": x.max(),
                    "Mean năm": x.mean() * TRADING_DAYS, "Std năm": x.std(ddof=1) * np.sqrt(TRADING_DAYS)}
    return pd.DataFrame(rows).T


desc = describe_returns(log_ret)
display(desc.astype(float).round(5))
for tk, row in desc.iterrows():
    print(f"{tk}: S = {row['Skewness']:+.3f} ({'lệch trái' if row['Skewness'] < 0 else 'lệch phải'}), "
          f"K = {row['Kurtosis']:.3f} ({'đuôi dày hơn chuẩn' if row['Kurtosis'] > 3 else 'đuôi mỏng'})")

# Histogram so với phân phối chuẩn cùng mean & std
fig, axes = grid_axes(N)
for ax, tk in zip(axes, tickers):
    x = log_ret[tk]
    ax.hist(x, bins=60, density=True, alpha=0.6, label="Thực tế")
    xs = np.linspace(x.min(), x.max(), 400)
    ax.plot(xs, stats.norm.pdf(xs, x.mean(), x.std()), "r-", lw=1.5, label="Chuẩn N(μ, σ²)")
    ax.set_title(f"{tk} | S = {desc.loc[tk, 'Skewness']:.2f}, K = {desc.loc[tk, 'Kurtosis']:.2f}")
    ax.grid(alpha=0.3); ax.legend(fontsize=8)
fig.suptitle("Phân phối log return so với phân phối chuẩn", fontsize=13, y=1.0)
plt.tight_layout(); plt.savefig(OUT["TV1"] / "1_histogram.png", dpi=150, bbox_inches="tight"); plt.show()

## 1.4b Kiểm định phân phối chuẩn: Jarque-Bera & QQ-plot
**Tầng 1:** $JB = \dfrac{n}{6}\left(S^2 + \dfrac{(K-3)^2}{4}\right) \sim \chi^2(2)$ dưới $H_0$: lợi suất phân phối chuẩn (S = 0, K = 3).

**Tầng 2:** `jarque_bera(x)` tính đúng công thức trên, đối chiếu với `scipy.stats.jarque_bera`. QQ-plot: phân vị mẫu (đã chuẩn hoá) so với phân vị N(0,1).

**Tầng 3:** p < 0,05 ⇒ bác bỏ chuẩn. Trên QQ-plot, hai đầu điểm **cong ra ngoài** đường 45° (dưới đường ở bên trái, trên đường ở bên phải)
⇒ giá trị cực đoan nhiều hơn chuẩn dự báo = fat tails.

**Lưu ý – biên độ giá ±7% của HOSE:** lợi suất ngày bị chặn ở trần/sàn nên histogram có cột nhô lên ở ≈ ±0,07 và QQ-plot "đi ngang"
ở hai đầu (rõ ở HPG, MWG, VHM). Biên độ **cắt cụt đuôi**: rủi ro đuôi thật (khi trần/sàn bị dồn sang nhiều phiên liên tiếp) có thể còn
lớn hơn những gì Kurtosis đo được ⇒ càng củng cố kết luận đuôi dày.

In [ ]:
# ===================== 1.4b JARQUE-BERA + QQ-PLOT =====================
def jarque_bera(x):
    x = np.asarray(x, float); x = x[~np.isnan(x)]
    n = len(x)
    S = stats.skew(x, bias=True)
    K = stats.kurtosis(x, fisher=False, bias=True)
    jb = n / 6 * (S ** 2 + (K - 3) ** 2 / 4)
    return n, S, K, jb, stats.chi2.sf(jb, df=2)


crit_jb = stats.chi2.ppf(1 - ALPHA, df=2)
jb_table = pd.DataFrame({tk: dict(zip(["n", "Skewness", "Kurtosis", "JB", "p-value"], jarque_bera(log_ret[tk])))
                         for tk in tickers}).T
jb_table["JB (scipy)"] = [stats.jarque_bera(log_ret[tk]).statistic for tk in tickers]
jb_table["Giá trị tới hạn χ²(2)"] = crit_jb
jb_table["Kết luận"] = np.where(jb_table["p-value"].astype(float) < ALPHA, "Bác bỏ H0 – KHÔNG chuẩn", "Chưa bác bỏ – chuẩn")
display(jb_table)

fig, axes = grid_axes(N, h=4, w=11)
for ax, tk in zip(axes, tickers):
    z = (log_ret[tk] - log_ret[tk].mean()) / log_ret[tk].std()
    (osm, osr), _ = stats.probplot(z, dist="norm")
    ax.scatter(osm, osr, s=6, alpha=0.6)
    lim = [min(osm.min(), osr.min()), max(osm.max(), osr.max())]
    ax.plot(lim, lim, "r-", lw=1.2, label="Đường 45° (chuẩn)")
    ax.set_title(f"QQ-plot – {tk}"); ax.set_xlabel("Phân vị N(0,1)"); ax.set_ylabel("Phân vị mẫu")
    ax.grid(alpha=0.3); ax.legend(fontsize=8)
fig.suptitle("QQ-plot log return so với phân phối chuẩn", fontsize=13, y=1.0)
plt.tight_layout(); plt.savefig(OUT["TV1"] / "1_qqplot.png", dpi=150, bbox_inches="tight"); plt.show()

## 1.4c Kiểm định tính dừng: ADF & KPSS (cho cả chuỗi giá và chuỗi lợi suất)
**Tầng 1:**
- ADF: $\Delta y_t = c + \gamma y_{t-1} + \sum_{j=1}^{p}\phi_j \Delta y_{t-j} + u_t$; $H_0: \gamma = 0$ (có nghiệm đơn vị – **không dừng**).
- KPSS: $H_0$: chuỗi **dừng**. Dùng hai kiểm định có giả thuyết ngược nhau để kết luận chắc chắn hơn.

**Tầng 2:** `adf_test`, `kpss_test` dùng `statsmodels` (nếu có), nếu không dùng bản viết tay (ADF chọn trễ theo AIC; KPSS dùng
phương sai dài hạn Newey-West). KPSS: bảng tra chỉ có p trong [0,01; 0,10] nên p bị chặn ở biên ⇒ cột "KPSS p (ghi báo cáo)"
ghi đúng dạng "< 0.01" / "> 0.10" để chép vào báo cáo.

**Tầng 3:** Kết luận DỪNG khi ADF bác bỏ (p < 0,05) **và** KPSS không bác bỏ (p ≥ 0,05). Kỳ vọng: giá KHÔNG DỪNG, lợi suất DỪNG.

In [ ]:
# ===================== 1.4c ADF & KPSS =====================
def _adf_manual(x, maxlag=None):
    """ADF có hằng số, chọn số trễ theo AIC. Trả về (stat, p=NaN, lag, giá trị tới hạn 5% xấp xỉ MacKinnon)."""
    x = np.asarray(x, float)
    dx = np.diff(x)
    maxlag = maxlag or int(np.ceil(12 * (len(x) / 100) ** 0.25))
    best = None
    for p in range(maxlag + 1):
        y = dx[maxlag:]
        X = [np.ones_like(y), x[maxlag:-1]] + [dx[maxlag - j:-j] for j in range(1, p + 1)]
        X = np.column_stack(X)
        beta, *_ = np.linalg.lstsq(X, y, rcond=None)
        resid = y - X @ beta
        n = len(y)
        aic = n * np.log(resid @ resid / n) + 2 * X.shape[1]
        if best is None or aic < best[0]:
            s2 = resid @ resid / (n - X.shape[1])
            se = np.sqrt(s2 * np.linalg.inv(X.T @ X)[1, 1])
            best = (aic, beta[1] / se, p)
    tau = best[1]
    p_approx = 0.01 if tau < -3.43 else 0.05 if tau < -2.86 else 0.10 if tau < -2.57 else 0.5
    return tau, p_approx, best[2], -2.86


def _kpss_manual(x):
    """KPSS (level) với phương sai dài hạn Bartlett, số trễ = int(12 (T/100)^{1/4})."""
    x = np.asarray(x, float)
    e = x - x.mean()
    n = len(e)
    L = int(12 * (n / 100) ** 0.25)
    s2 = e @ e / n
    for k in range(1, L + 1):
        s2 += 2 * (1 - k / (L + 1)) * (e[k:] @ e[:-k]) / n
    stat = (np.cumsum(e) ** 2).sum() / (n ** 2 * s2)
    p = float(np.interp(stat, [0.347, 0.463, 0.574, 0.739], [0.10, 0.05, 0.025, 0.01]))
    return stat, p, L, 0.463


def adf_test(x):
    x = pd.Series(x).dropna()
    if HAS_SM:
        stat, p, lag, _, cv, _ = adfuller(x, regression="c", autolag="AIC")
        return {"ADF stat": stat, "ADF p-value": p, "ADF lag": lag, "ADF CV 5%": cv["5%"]}
    stat, p, lag, cv = _adf_manual(x.values)
    return {"ADF stat": stat, "ADF p-value": p, "ADF lag": lag, "ADF CV 5%": cv}


def kpss_test(x):
    x = pd.Series(x).dropna()
    if HAS_SM:
        with warnings.catch_warnings():
            warnings.simplefilter("ignore", InterpolationWarning)
            stat, p, lag, cv = kpss(x, regression="c", nlags="auto")
    else:
        stat, p, lag, cv = _kpss_manual(x.values)
        cv = {"5%": cv}
    # bảng tra KPSS chỉ có p trong [0,01; 0,10] -> p ở biên nghĩa là "< 0,01" hoặc "> 0,10"
    note = "< 0.01" if p <= 0.01 else ("> 0.10" if p >= 0.10 else f"{p:.3f}")
    return {"KPSS stat": stat, "KPSS p-value": p, "KPSS p (ghi báo cáo)": note, "KPSS lag": lag, "KPSS CV 5%": cv["5%"]}


def stationarity_verdict(p_adf, p_kpss):
    adf_rej, kpss_rej = p_adf < ALPHA, p_kpss < ALPHA
    if adf_rej and not kpss_rej:
        return "DỪNG"
    if not adf_rej and kpss_rej:
        return "KHÔNG DỪNG"
    return "Mâu thuẫn (cần xem thêm)"


rows = []
for name, df in {"Giá (Adj Close)": prices, "Log return": log_ret}.items():
    for tk in tickers:
        res = {"Chuỗi": name, "Mã": tk, **adf_test(df[tk]), **kpss_test(df[tk])}
        res["Kết luận"] = stationarity_verdict(res["ADF p-value"], res["KPSS p-value"])
        rows.append(res)
stationarity = pd.DataFrame(rows).set_index(["Chuỗi", "Mã"])
display(stationarity.round(4))

## 1.4d Kiểm định tự tương quan: ACF & Ljung-Box trên $r_t$, $|r_t|$, $r_t^2$
**Tầng 1:** $Q(m) = T(T+2)\sum_{k=1}^{m}\dfrac{\hat\rho_k^2}{T-k} \sim \chi^2(m)$ dưới $H_0$: không có tự tương quan đến trễ m.
$r_t$ có tự tương quan ⇒ lợi suất dự báo được (vi phạm thị trường hiệu quả dạng yếu); $|r_t|$, $r_t^2$ có tự tương quan ⇒ **cụm biến động** (hiệu ứng ARCH).

**Tầng 2:** `acf()` và `ljung_box()` viết tay bằng numpy (kết quả trùng `statsmodels.acorr_ljungbox`). Biên tin cậy ACF: $\pm 1{,}96/\sqrt{T}$.

**Tầng 3:** Cột ACF vượt dải xanh = tự tương quan có ý nghĩa. Ô p-value < 0,05 = có tự tương quan.

In [ ]:
# ===================== 1.4d ACF + LJUNG-BOX =====================
def acf(x, nlags):
    """Hệ số tự tương quan mẫu ρ_1..ρ_nlags."""
    x = np.asarray(x, float); x = x - x.mean()
    denom = x @ x
    return np.array([(x[k:] @ x[:-k]) / denom for k in range(1, nlags + 1)])


def ljung_box(x, lags):
    """Ljung-Box Q(m) và p-value cho từng m trong lags."""
    x = np.asarray(pd.Series(x).dropna(), float)
    n = len(x)
    rho = acf(x, max(lags))
    out = {}
    for m in lags:
        Q = n * (n + 2) * np.sum(rho[:m] ** 2 / (n - np.arange(1, m + 1)))
        out[m] = {"Q": Q, "p-value": stats.chi2.sf(Q, m)}
    return pd.DataFrame(out).T


ACF_LAGS = 30
transforms = {"r_t": lambda s: s, "|r_t|": lambda s: s.abs(), "r_t²": lambda s: s ** 2}
band = 1.96 / np.sqrt(T)
fig, axes = plt.subplots(N, 3, figsize=(15, 2.6 * N), squeeze=False, sharey=True)
for i, tk in enumerate(tickers):
    for j, (name, f) in enumerate(transforms.items()):
        ax = axes[i, j]
        rho = acf(f(log_ret[tk]), ACF_LAGS)
        ax.vlines(range(1, ACF_LAGS + 1), 0, rho, color="tab:blue")
        ax.scatter(range(1, ACF_LAGS + 1), rho, s=10, color="tab:blue")
        ax.axhspan(-band, band, color="tab:blue", alpha=0.12)
        ax.axhline(0, color="black", lw=0.5)
        ax.set_title(f"{tk} – ACF của {name}", fontsize=10); ax.grid(alpha=0.3)
fig.suptitle("ACF của r_t, |r_t|, r_t² (dải xanh = khoảng tin cậy 95%)", fontsize=13, y=1.0)
plt.tight_layout(); plt.savefig(OUT["TV1"] / "1_acf.png", dpi=150, bbox_inches="tight"); plt.show()

rows = []
for tk in tickers:
    for name, f in transforms.items():
        for m, r in ljung_box(f(log_ret[tk]), LB_LAGS).iterrows():
            rows.append({"Mã": tk, "Chuỗi": name, "Lag": int(m), "Q-stat": r["Q"], "p-value": r["p-value"],
                         "Kết luận": "Có tự tương quan" if r["p-value"] < ALPHA else "Không"})
ljungbox = pd.DataFrame(rows)
lb_pivot = ljungbox.pivot_table(index=["Mã", "Chuỗi"], columns="Lag", values="p-value", sort=False)
lb_pivot.columns = [f"p (lag {c})" for c in lb_pivot.columns]
display(lb_pivot.round(4))

## 1.5 Diễn giải cốt lõi: Fat Tails & Volatility Clustering ⇒ vì sao cần GARCH
**Tầng 1:**
- *Fat tails*: K > 3 và JB bác bỏ chuẩn ⇒ biến động cực đoan xảy ra thường xuyên hơn phân phối chuẩn dự báo ⇒ VaR giả định chuẩn
  với phương sai cố định sẽ **đánh giá thấp** rủi ro đuôi.
- *Volatility clustering*: $|r_t|$ và $r_t^2$ tự tương quan rất mạnh (p ≈ 0) ⇒ phương sai **thay đổi theo thời gian và phụ thuộc quá khứ**.
- Bản thân $r_t$ tự tương quan yếu hơn nhiều; nếu mã nào vẫn có p < 0,05 (ô tổng hợp dưới tự chỉ ra) thì thường do **giao dịch mỏng**
  (nhiều ngày lợi suất = 0 do điền giá) ⇒ có thể thêm AR(1) vào phương trình trung bình; điều này không thay đổi kết luận chính.
- Mô hình phương sai không đổi không giải thích được các hiện tượng trên ⇒ cần **GARCH** $\sigma_t^2 = \omega + \alpha\epsilon_{t-1}^2 + \beta\sigma_{t-1}^2$
  (TV2), kết hợp phân phối Student-t để bắt đuôi dày.

**Tầng 3:** ô dưới tự tổng hợp bằng chứng cho từng mã từ các bảng trên.

In [ ]:
# ===================== 1.5 TỔNG HỢP STYLIZED FACTS =====================
max_lag = max(LB_LAGS)
lb_at = lambda name: ljungbox[(ljungbox["Chuỗi"] == name) & (ljungbox["Lag"] == max_lag)].set_index("Mã")["p-value"]
evidence = pd.DataFrame(index=tickers)
evidence["Kurtosis"] = desc["Kurtosis"].astype(float)
evidence["JB p-value"] = jb_table["p-value"].astype(float)
evidence["Fat tails?"] = np.where((evidence["Kurtosis"] > 3) & (evidence["JB p-value"] < ALPHA), "CÓ", "không")
evidence[f"LB p r_t (lag {max_lag})"] = lb_at("r_t")
evidence[f"LB p r_t² (lag {max_lag})"] = lb_at("r_t²")
evidence["Vol clustering?"] = np.where(lb_at("r_t²") < ALPHA, "CÓ", "không")
evidence["Tự tương quan r_t?"] = np.where(lb_at("r_t") < ALPHA, "CÓ (nhẹ)", "không")   # kết luận theo lag 20
sig_lags = ljungbox[(ljungbox["Chuỗi"] == "r_t") & (ljungbox["p-value"] < ALPHA)].groupby("Mã")["Lag"].apply(
    lambda s: ", ".join(map(str, s)))
evidence["LB r_t có ý nghĩa ở lag"] = sig_lags.reindex(tickers).fillna("không lag nào")
evidence["% ngày r_t = 0"] = (log_ret == 0).mean() * 100
evidence["Giá"] = stationarity.xs("Giá (Adj Close)", level="Chuỗi")["Kết luận"]
evidence["Lợi suất"] = stationarity.xs("Log return", level="Chuỗi")["Kết luận"]
display(evidence.round(4))

print(f"• Fat tails ở {(evidence['Fat tails?'] == 'CÓ').sum()}/{N} mã -> dùng phân phối Student-t (TV2) và VaR-t (TV4).")
print(f"• Volatility clustering ở {(evidence['Vol clustering?'] == 'CÓ').sum()}/{N} mã -> cần GARCH/GJR-GARCH (TV2).")
print("• Giá: " + ", ".join(f"{k}={v}" for k, v in evidence["Giá"].items()))
print("• Lợi suất: " + ", ".join(f"{k}={v}" for k, v in evidence["Lợi suất"].items())
      + " -> mọi mô hình phía sau ước lượng trên log return.")
ac_tk = evidence.index[evidence["Tự tương quan r_t?"] != "không"].tolist()
if ac_tk:
    print(f"• Lợi suất còn tự tương quan nhẹ ở {', '.join(ac_tk)} (tỷ lệ ngày r_t = 0: "
          + ", ".join(f"{t} = {evidence.loc[t, '% ngày r_t = 0']:.1f}%" for t in ac_tk)
          + ") -> nghi do giao dịch mỏng; kiểm tra lại Ljung-Box trên phần dư ε ở TV2.")
weak = [t for t in tickers if t not in ac_tk and evidence.loc[t, "LB r_t có ý nghĩa ở lag"] != "không lag nào"]
if weak:
    print("• Tự tương quan rất yếu (chỉ có ý nghĩa ở một vài lag, không có ý nghĩa ở lag 20): "
          + ", ".join(f"{t} (lag {evidence.loc[t, 'LB r_t có ý nghĩa ở lag']})" for t in weak)
          + " -> kết luận ở bảng tổng hợp dựa trên lag 20.")

with pd.ExcelWriter(OUT["TV1"] / "TV1_statistical_tests.xlsx", engine="openpyxl") as xw:
    evidence.to_excel(xw, sheet_name="Tong_hop")
    desc.to_excel(xw, sheet_name="Thong_ke_mo_ta")
    jb_table.to_excel(xw, sheet_name="Jarque_Bera")
    stationarity.to_excel(xw, sheet_name="ADF_KPSS")
    ljungbox.to_excel(xw, sheet_name="Ljung_Box", index=False)
print("✔ Đã lưu", OUT["TV1"] / "TV1_statistical_tests.xlsx")

### TV1 – Gợi ý trả lời vấn đáp
**Câu 1 – Vì sao log return dừng còn giá thì không?** Giá xấp xỉ bước ngẫu nhiên $\ln P_t = \ln P_{t-1} + r_t$ (có nghiệm đơn vị):
mọi cú sốc tích luỹ vĩnh viễn nên trung bình và phương sai của giá thay đổi theo thời gian. Log return chính là sai phân bậc 1 của
$\ln P_t$, loại bỏ nghiệm đơn vị nên dao động quanh một mức trung bình ổn định ⇒ dừng (ADF bác bỏ, KPSS không bác bỏ).

**Câu 2 – Kurtosis > 3 và JB p < 0,05 nghĩa là gì cho việc chọn mô hình rủi ro?** Lợi suất không chuẩn và đuôi dày ⇒ VaR/ES giả định
chuẩn đánh giá thấp tổn thất cực đoan (nhất là ở 99%). Cần: (i) mô hình phương sai có điều kiện (GARCH) vì một phần đuôi dày do phương sai
thay đổi; (ii) phân phối đuôi dày (Student-t); (iii) bổ sung ES và Historical Simulation, rồi backtest (TV4) để kiểm chứng.